In [1]:
library(car)  # For Type III ANOVA
load("/nfs/dcmb-lgarmire/yhdu/COBRE Final Documents/RNA-seq analysis/RNA_normalized_log_counts.rdata",verbose=TRUE)
cobre_pd = readRDS("/nfs/dcmb-lgarmire/yhdu/COBRE Final Documents/Figure 1 Differential analysis/cobre_pd.rds")
rownames(cobre_pd) = cobre_pd$Sample_Name

Loading required package: carData



Loading objects:
  pd
  dds
  normalized_counts
  lognormalized_counts


In [2]:
common_id = intersect(colnames(lognormalized_counts),rownames(cobre_pd))
sovdat = t(lognormalized_counts[,common_id])
pd = cobre_pd[common_id,c('Sample_Group','Mat_Ethnicity','Mat_Age','Net_Weight_Gain',
                          'Gravidity','Parity','Pat_Ethnicity','Gestational_Age','Hemoglobin','Sex')]

In [3]:
probe_names <- colnames(sovdat)
sovdatlist <- as.data.frame(sovdat)
pd$beta <- NA
Ftab <- data.frame(matrix(nrow = length(probe_names), ncol = ncol(pd) - 1))
colnames(Ftab) <- colnames(pd)[-ncol(pd)]  # Exclude 'beta'
rownames(Ftab) <- probe_names

In [4]:
# Identify probes with all zeros or constant values
constant_probes <- apply(sovdat, 2, function(x) var(x) == 0)
sum(constant_probes)

[1] 0

In [5]:
probe <- sovdat[,1]
newdata <- pd
newdata$beta <- probe  # Replace the beta column with current probe values

In [7]:
formula <- as.formula(paste0("beta ~ ", paste0(names(newdata)[-ncol(newdata)], collapse = " + ")))
formula

beta ~ Sample_Group + Mat_Ethnicity + Mat_Age + Net_Weight_Gain + 
    Gravidity + Parity + Pat_Ethnicity + Gestational_Age + Hemoglobin + 
    Sex

In [5]:
calF <- function(probe) {
  newdata <- pd
  newdata$beta <- probe  # Replace the beta column with current probe values
    
  # Construct the formula for linear model
  formula <- as.formula(paste0("beta ~ ", paste0(names(newdata)[-ncol(newdata)], collapse = " + ")))
  
  result <- tryCatch({
    fit <- lm(formula, data = newdata)
    aovfit <- Anova(fit, type = 3, singular.ok = TRUE)
    
    # Extract F-statistics (exclude intercept row)
    F_values <- aovfit$`F value`[-1][1:10]
    names(F_values) <- names(newdata)[-ncol(newdata)]
    return(F_values)
  }, error = function(e) {
    return(rep(NA, length(names(newdata)[-ncol(newdata)])))  # Return NAs if an error occurs
  })
  return(result)
}

In [52]:
for (i in 1:length(probe_names)) {
  probe <- sovdat[,i]  # Get probe beta values
  Ftab[i, ] <- calF(probe)  # Store F-statistics
}

In [54]:
Ftab[1:5,]

,Sample_Group,Mat_Ethnicity,Mat_Age,Net_Weight_Gain,Gravidity,Parity,Pat_Ethnicity,Gestational_Age,Hemoglobin,Sex
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
DDX11L1,0.084622997,1.09861242,0.005444634,4.601465,1.0586097,1.6032091,0.33964664,0.01839737,3.89237968,0.6255514
WASH7P,0.045726230,1.20216881,1.830347425,1.354397,1.8322982,0.5942408,0.32260420,3.09375242,4.70115657,0.5596502
MIR6859.1,1.465325649,0.38688737,2.661578151,5.278091,0.3242985,0.1126131,0.01003808,0.01751460,0.08198753,0.9031953
MIR1302.2HG,0.022649179,1.65054631,0.559319308,0.653782,0.3080220,0.7723349,0.31608694,0.57135540,0.29187186,2.2103977
OR4G11P,0.005378074,0.06679194,0.010515902,0.795374,0.6735579,1.3893853,0.46526511,0.70322679,0.30292341,0.9777785


In [55]:
saveRDS(Ftab,'Ftab_COBRE_GENE_EXPR.rds')

In [59]:
Fmean <- colMeans(Ftab, na.rm = TRUE)
Fmean <- Fmean[order(-Fmean)]  # Sort in descending order

# Create final output
Fmean_df <- data.frame(Factor = names(Fmean), Fstat = as.vector(Fmean), stringsAsFactors = FALSE)

# Select significant factors (e.g., F > 1)
finalvars <- c('Sample_Group', Fmean_df$Factor[Fmean_df$Fstat > 1])

saveRDS(Fmean_df,'Fmean_df_cobre_gene_expression.rds')
saveRDS(finalvars,'finalvars_cobre_gene_expression.rds')
# Print results
print("Gene expression Mean F-statistics for each factor:")
(Fmean_df)
print("Gene expression Selected variables with significant variance explained:")
print(finalvars)

[1] "Gene expression Mean F-statistics for each factor:"


Factor,Fstat
<chr>,<dbl>
Mat_Age,1.4972013
Sex,1.3300179
Hemoglobin,1.1970837
Sample_Group,1.1590262
Net_Weight_Gain,1.0792836
Mat_Ethnicity,1.0399727
Gravidity,1.0171646
Parity,1.0153954
Gestational_Age,0.9559303


[1] "Gene expression Selected variables with significant variance explained:"
[1] "Sample_Group"    "Mat_Age"         "Sex"             "Hemoglobin"     
[5] "Sample_Group"    "Net_Weight_Gain" "Mat_Ethnicity"   "Gravidity"      
[9] "Parity"         


In [65]:
library(ggplot2)
Fmean_df <- Fmean_df[order(-Fmean_df$Fstat), ]
Fstat_data = Fmean_df
png('sov_gene_expr.png',width=7,height=6,res=300,unit='in')
ggplot(Fstat_data, aes(x = reorder(Factor, -Fstat), y = Fstat, fill = Factor)) +
  geom_bar(stat = "identity") +
  geom_hline(yintercept = 1, linetype = "dotted", color = "red", size = 1) +  # Add horizontal threshold line
  labs(
    title = "Source of Variance (Gene Expression)",
    y = "F-statistics",
    x = ""
  ) +
    ylim(0, 2) + 
  theme_minimal(base_size = 14) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),  # Rotate x-axis labels
    legend.position = "none",  # Remove legend
    plot.title = element_text(face = "bold", size = 16, hjust = 0.5)
  )
dev.off()

png 
  2